### RSS 수집기

In [1]:
# 라이브러리 불러오기
import feedparser
import yfinance as yf
from datetime import datetime, timedelta
import pandas as pd
import hashlib
import json

In [18]:
RSS_SOURCES = {
    "politics_economy": [
        "https://www.hankyung.com/feed/economy",                        # 한국경제 50개
        "https://www.yna.co.kr/rss/economy.xml",                        # 연합뉴스 경제 120개
        "https://www.yna.co.kr/rss/politics.xml",                       # 연합뉴스 정치 120개
        "https://rss.donga.com/economy.xml",                            # 동아일보 경제 50개
        "https://rss.donga.com/politics.xml",                           # 동아일보 정치 50개
        "https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=02",  # SBS 경제 29개
        "https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=01",  # SBS 정치 29개
        "https://www.khan.co.kr/rss/rssdata/politic_news.xml",         # 경향 정치 50개
        "https://www.asiae.co.kr/rss/economy.htm",                      # 아시아경제 100개
    ],
    "global_macro": [
        "https://feeds.feedburner.com/businessinsider",                 # Business Insider 50개
        "https://rss.nytimes.com/services/xml/rss/nyt/Economy.xml",    # NYT 경제 20개
        "https://www.investing.com/rss/news.rss",                       # Investing.com 10개
        "https://www.investing.com/rss/market_overview.rss",            # Investing.com 매크로 10개
        "https://feeds.marketwatch.com/marketwatch/topstories/",        # MarketWatch 10개
        "https://www.cnbc.com/id/100003114/device/rss/rss.html",       # CNBC 30개
        "https://www.cnbc.com/id/10000664/device/rss/rss.html",        # CNBC 경제 30개
        "https://www.hankyung.com/feed/international",                  # 한경 국제 50개
        "https://www.yna.co.kr/rss/international.xml",                  # 연합 국제 120개
        "https://www.wired.com/feed/rss",                               # Wired 50개
    ],
    "tech_industry": [
        "https://techcrunch.com/feed/",                                 # TechCrunch 20개
        "https://feeds.feedburner.com/venturebeat/SZYF",                # VentureBeat 7개
        "https://rss.etnews.com/Section901.xml",                        # 전자신문 IT 30개
        "https://rss.etnews.com/Section902.xml",                        # 전자신문 반도체 30개
        "https://rss.etnews.com/Section903.xml",                        # 전자신문 디스플레이 20개
        "https://feeds.arstechnica.com/arstechnica/index",              # Ars Technica 20개
    ],
    "real_estate": [
        "https://www.hankyung.com/feed/realestate",                     # 한경 부동산 50개
        "https://www.asiae.co.kr/rss/realestate.htm",                  # 아시아경제 부동산 41개
        "https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=07",  # SBS 부동산 29개
        "https://www.mk.co.kr/rss/50200011/",                          # 매경 부동산 50개
        "https://www.mk.co.kr/rss/50100032/",                          # 매경 부동산2 50개
    ],
}

In [19]:
# RSS 수집 함수
def collect_rss(category: str, days: int = 7) -> list:
    '''
    카테고리별 RSS 수집
    days: 과거 며칠치 수집 할지
    '''
    results = []
    since = datetime.now() - timedelta(days=days)
    urls = RSS_SOURCES.get(category, [])

    for url in urls:
        try:
            feed = feedparser.parse(url)
            print(f'{url} → {len(feed.entries)}개 수집')

            for entry in feed.entries:
                # 날짜 파싱
                published = entry.get('published_parsed')
                if published:
                    pub_date = datetime(*published[:6])
                    if pub_date < since:
                        continue
                # 중복 제거용 해시
                url_hash = hashlib.md5(entry.get('link', '').encode()).hexdigest()        
                
                results.append({
                    'hash': url_hash,
                    'category': category,
                    'title': entry.get('title', ''),
                    'link': entry.get('link', ''),
                    'summary': entry.get('summary', ''),
                    'published': entry.get('published', ''),
                    'source': url,
                })

        except Exception as e:
            print(f'Error parsing {url}: {e}')
    
    print(f'{category} 카테고리 {len(results)}개 수집 완료')
    return results

In [20]:
# 테스트

# 정치/경제 수집
articles = collect_rss('politics_economy', days=7)

# 결과 확인
df = pd.DataFrame(articles)
print(df[['title', 'published','source']].head(10))

https://www.hankyung.com/feed/economy → 50개 수집
https://www.yna.co.kr/rss/economy.xml → 120개 수집
https://www.yna.co.kr/rss/politics.xml → 120개 수집
https://rss.donga.com/economy.xml → 50개 수집
https://rss.donga.com/politics.xml → 50개 수집
https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=02 → 29개 수집
https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=01 → 29개 수집
https://www.khan.co.kr/rss/rssdata/politic_news.xml → 50개 수집
https://www.asiae.co.kr/rss/economy.htm → 100개 수집
politics_economy 카테고리 598개 수집 완료
                                               title  \
0                     "역대급 더위 온다"…삼성전자, 에어컨 생산라인 풀가동   
1                "성과급 40조 달라"는 삼성전자 노조…400만 개미들 '한숨'   
2              [단독] '죽전 신세계' 40분간 암흑…비상전력 시스템마저 '먹통'   
3                   중동發 에너지 위기에도…"재생에너지로의 즉각 전환 어려워"   
4            "요즘 이만한 차가 어딨어요"…'연비 최강' 이 車 뭐길래 [신차털기]   
5         대만 국민당의 위험한 승부수…국공회담을 통한 정리원의 노림수 [차이나 워치]   
6  금융위, 우수 직원에 최대 1000만원 상금…'명예의 전당'도 만든다 [금융당국 백...   
7                프랜차이즈 가맹본부 13.2% 증가…주춤했던

In [22]:
# 전체 카테고리 수집
def collect_all(days: int = 7) -> dict:
    '''전체 카테고리 수집'''
    all_categories = {}
    for category in RSS_SOURCES.keys():
        print(f'{category}수집 중')
        all_categories[category] = collect_rss(category, days)

    total = sum(len(v) for v in all_categories.values())
    print(f'총 {total}개 뉴스 수집 완료')
    return all_categories

# 실행
all_articles = collect_all(days=7)

politics_economy수집 중
https://www.hankyung.com/feed/economy → 50개 수집
https://www.yna.co.kr/rss/economy.xml → 120개 수집
https://www.yna.co.kr/rss/politics.xml → 120개 수집
https://rss.donga.com/economy.xml → 50개 수집
https://rss.donga.com/politics.xml → 50개 수집
https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=02 → 29개 수집
https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=01 → 29개 수집
https://www.khan.co.kr/rss/rssdata/politic_news.xml → 50개 수집
https://www.asiae.co.kr/rss/economy.htm → 100개 수집
politics_economy 카테고리 598개 수집 완료
global_macro수집 중
https://feeds.feedburner.com/businessinsider → 50개 수집
https://rss.nytimes.com/services/xml/rss/nyt/Economy.xml → 20개 수집
https://www.investing.com/rss/news.rss → 10개 수집
https://www.investing.com/rss/market_overview.rss → 10개 수집
https://feeds.marketwatch.com/marketwatch/topstories/ → 10개 수집
https://www.cnbc.com/id/100003114/device/rss/rss.html → 30개 수집
https://www.cnbc.com/id/10000664/device/rss/rss.html → 30개 수집
https://www.hankyung.com/feed/intern

In [23]:
# 결과 저장
with open('../data/raw_articles.json', 'w', encoding='utf-8') as f:
    json.dump(all_articles, f, ensure_ascii=False, indent=2)

print("저장 완료")

저장 완료
